# CPTR 480 — Real-Time Embedded Systems
## Spring 2026 · Week 1, Day 1

**Prof. Frohne · March 30, 2026**

---

> *By Week 9, this board in my hand will be a working USB software-defined radio receiver — tunable from your laptop in real time.*

---

### Today
1. What makes a system "real-time"?
2. Why C/C++ and not Python?  *(we'll do the math)*
3. AI-assisted development — Copilot as a tool, not an oracle
4. Environment setup + Lab 1 preview

---
## What Does "Real-Time" Actually Mean?

### Hint: it does NOT mean *fast*.

> **Real-time = predictable.** The system must complete its task within a *deadline*, every single time.

| Category | Miss a deadline and… | Example |
|---|---|---|
| **Hard real-time** | System failure / danger | Airbag controller, pacemaker |
| **Firm real-time** | Result is worthless | Video frame decoder |
| **Soft real-time** | Degraded experience | Music streaming buffer |

---

### Discussion: What real-time systems did you interact with *today*?

*(traffic lights · antilock brakes · touchscreen · audio playback · phone camera autofocus)*

---
## Our Application: A 48 kHz Audio ADC over USB

The **PCM1808** on your 2026 board captures audio at:
- **48,000 samples/second** · stereo · 24-bit

That means the system must be ready for a new sample every:

$$T_{sample} = \frac{1}{48{,}000} \approx 20.8\,\mu s$$

**Miss that deadline → audio dropout.** The USB host on your laptop expects continuous data and will click, pop, or mute.

Let's calculate this — and the total data rate — right now.

In [1]:
## Audio timing — let's do the math

sample_rate   = 48_000      # samples/sec
channels      = 2           # stereo
bits_per_sample = 24        # PCM1808 output
bytes_per_sample = bits_per_sample // 8

sample_interval_us = 1e6 / sample_rate
data_rate_kBps     = (sample_rate * channels * bytes_per_sample) / 1024

print(f"Sample interval : {sample_interval_us:.1f} µs")
print(f"Data rate       : {data_rate_kBps:.1f} kB/s  ({data_rate_kBps/1024:.3f} MB/s)")
print()
print("The firmware has 20.8 µs to be ready for each sample — every time, with no exceptions.")

Sample interval : 20.8 µs
Data rate       : 281.2 kB/s  (0.275 MB/s)

The firmware has 20.8 µs to be ready for each sample — every time, with no exceptions.


---
## Why Not MicroPython?

MicroPython is great for prototyping. But it has one hard showstopper for continuous audio:

| Problem | Impact on 20.8 µs deadline |
|---|---|
| Bytecode interpreter: ~1–10 µs **per instruction** | Near the budget just to execute one line |
| **Garbage collector pauses: 10–50 ms** | Pauses your audio callback mid-execution. Audible pop/glitch. |
| GC can strike *even when DMA is running* | DMA keeps filling buffers; Python stops reading them → overrun |

> **Clarification:** `machine.I2S` in MicroPython *does* use DMA internally for the hardware transfer — so data moves from the peripheral to a buffer without CPU polling. But GC can pause the **Python callback** that processes that buffer. DMA keeps filling the next buffer while Python is frozen, and by the time GC finishes, data has been overwritten. The hardware path is fine; the software path is the problem.

In [2]:
## How bad is a GC pause?

gc_pause_ms_min = 10   # ms — optimistic GC pause in MicroPython
gc_pause_ms_max = 50   # ms — realistic worst case

samples_dropped_min = gc_pause_ms_min * 1e-3 * sample_rate
samples_dropped_max = gc_pause_ms_max * 1e-3 * sample_rate

print(f"GC pause range     : {gc_pause_ms_min}–{gc_pause_ms_max} ms")
print(f"Samples dropped    : {samples_dropped_min:.0f}–{samples_dropped_max:.0f} samples")
print()
print("Human hearing detects clicks at gaps as short as ~1 ms (48 samples).")
print("A 10 ms GC pause is 10× that threshold. You WILL hear it.")

GC pause range     : 10–50 ms
Samples dropped    : 480–2400 samples

Human hearing detects clicks at gaps as short as ~1 ms (48 samples).
A 10 ms GC pause is 10× that threshold. You WILL hear it.


---
## The Solution: C/C++ + DMA

**DMA (Direct Memory Access)** moves data from the hardware peripheral to a memory buffer *without involving the CPU at all.*

```
WITH DMA (both MicroPython machine.I2S and C SDK)
  [I2S peripheral] --DREQ→ [DMA controller] --copy bytes→ [RAM buffer A]
                                               ↑ CPU wakes ONCE per buffer (~10 ms)
```

MicroPython *does* use DMA for I2S input. The difference is what happens **next**:

```
MicroPython (machine.I2S + DMA)
  DMA fills Buffer A ──────────────────────────────────────────→ ?
  Python callback fires to process A ──→ [GC PAUSE 10–50 ms] ──→ overrun!
  DMA has already filled Buffer B and started overwriting Buffer A.

C/C++ SDK + DMA (ping-pong)
  DMA fills Buffer A → ISR runs in SRAM (<1 µs) → hand off to Core 1
  DMA fills Buffer B → ISR runs in SRAM (<1 µs) → hand off to Core 1
  ...forever, no GC, no interpreter, no overruns
```

### Why no GC in C?

C/C++ has no garbage collector. Memory is managed explicitly. The ISR runs deterministically every time.**This is why we use C/C++ — not because MicroPython lacks DMA, but because MicroPython's runtime can't guarantee callback latency.**


---
## Why the Raspberry Pi Pico (RP2040)?

| Feature | Why it matters |
|---|---|
| **2× Cortex-M0+ cores** | Core 0 handles USB; Core 1 handles audio capture — no sharing |
| **No OS by default** | No scheduler jitter. Your code is the only thing running. |
| **DMA controller** (12 channels) | Continuous background data movement |
| **PIO (Programmable I/O)** | Implements I2S in 10 lines of assembly — no dedicated hardware needed |
| **264 KB SRAM, 6 banks** | Parallel access from both cores without bus contention |
| **~$4 retail** | You can afford to brick one while learning |

> The RP2040 has **no dedicated I2S peripheral**. Instead, PIO lets you synthesize any serial protocol in hardware. We'll study this in Week 2.

---
## AI-Assisted Development: Copilot as a Tool

GitHub Copilot is part of this course's workflow. Here's the honest picture:

### What it's good at
- Reading and summarizing datasheets (paste text, ask questions)
- Generating boilerplate: CMakeLists.txt, register init tables, pin assignment tables
- Explaining unfamiliar APIs: *"What does `dma_channel_configure()` do?"*
- First drafts of documentation

### What it gets wrong
- **Register names and bit fields** — it hallucinates these. Always verify against the actual datasheet.
- **Board-specific details** — it doesn't know your 2026 board unless you give it context
- **Subtle timing constraints** — it may suggest code that works most of the time but fails under load

### The mental model
> Copilot is a **senior developer who types fast but sometimes confidently lies.**  
> Your job: verify everything against the datasheet. The AI reflection journal is where you track what it got right and wrong.

### The `@workspace` trick  
In VS Code Copilot Chat, prefix your question with `@workspace` and Copilot will search *your actual code* for context — not just its training data.

---
## Copilot Demo — Try These Prompts

Open **Copilot Chat** in VS Code and try:

```
What is DMA and why is it used in embedded audio?
```

```
@workspace What CMake targets are in my project?
```

*(paste the first paragraph of the PCM1808 datasheet, then ask:)*
```
What is the master clock frequency this ADC requires at 48 kHz?
```

**Watch for:** does it get the MCLK ratio right (256×fs = 12.288 MHz)? If not, that's your first journal entry.

In [3]:
## Quick check: PCM1808 master clock requirement

fs = 48_000           # sample rate (Hz)
mclk_ratio = 256      # standard for PCM1808 at 48 kHz

mclk_hz = fs * mclk_ratio
mclk_mhz = mclk_hz / 1e6

print(f"Required MCLK = {mclk_ratio} × fs = {mclk_ratio} × {fs:,} = {mclk_hz:,} Hz = {mclk_mhz:.3f} MHz")
print()
print("The Si5351a clock generator on your board produces this frequency.")
print("Configuring it correctly is Lab 2, Part 4.")

Required MCLK = 256 × fs = 256 × 48,000 = 12,288,000 Hz = 12.288 MHz

The Si5351a clock generator on your board produces this frequency.
Configuring it correctly is Lab 2, Part 4.


Note:  This is incorrect for our 2026 board.  We use a 24.576 MHz clock, not a 12.288 MHz clock.  :-)  The AI got this wrong, but it is still useful.  You have to take everything I say, and everything the AI tells you with a grain of salt.  Trust, but verify!


---
## Environment Setup Checklist

Do these **now** (in parallel with the lecture):

- [ ] Install **VS Code** — [code.visualstudio.com](https://code.visualstudio.com)
- [ ] Install extensions: `C/C++`, `CMake Tools`, `MicroPico`, `GitHub Copilot`
- [ ] Sign into **GitHub** inside VS Code
- [ ] Submit your transcript at **[education.github.com](https://education.github.com)** to activate Copilot free tier
- [ ] Clone the class template repo *(URL on the board)*
- [ ] Open Copilot Chat — confirm it responds

**Windows users:** Use native Windows or Linux. WSL2 + USB device access for Pico flashing is unreliable.

---
## Lab 1 Preview — Tomorrow, March 31 (3 hours)

**What you'll do:**
1. Solder headers onto your 2026 board and Debug Stack
2. Verify solder joints with a multimeter (continuity mode)
3. Flash MicroPython and blink an LED — *find the GPIO number yourself from the schematic*
4. Capture the blink waveform with the logic analyzer
5. Build and flash the C/C++ SDK blink — verify your toolchain works

**Arrive having read the Lab 1 document.**  
Cold solder joints from rushing through Part 1 will cost you the rest of the lab.

---

## HW 1 — Due Monday April 6, 11:00 a.m.

- Read the **PCM1808** and **Si5351a** datasheets (linked in the template repo)
- Start your **2026 board README skeleton** (`assignment1/2026_board_README_draft.md`)
  - Pin assignment table (trace from the schematic)
  - Power rail expected values
  - Bring-up checklist stub

> **Tip:** Use Copilot to generate the pin table draft from the netlist — then audit it. That's the workflow.

---

*Questions? See you in lab tomorrow at 2:00 p.m.*